# Security Practical Lab

## 📓 Interactive Notebook · Security Module

In this notebook, you'll practice:
1. **Secrets management** with Streamlit
2. **Input validation** patterns
3. **SQL injection** prevention
4. **File upload** security
5. **LLM prompt injection** awareness

---

## 📋 Objectives

By the end of this notebook, you will be able to:
- Manage secrets securely
- Validate all user inputs
- Prevent common security vulnerabilities
- Handle file uploads safely
- Protect against prompt injection

---

## 🔐 Lab 1: Secrets Management

In [ ]:
import streamlit as st

st.header("🔐 Lab 1: Secrets Management")

st.subheader("❌ WRONG: Hard-coded secrets")
st.code('''
# NEVER do this!
API_KEY = "sk-1234567890abcdef"  # Exposed to everyone!
DB_PASSWORD = "mypassword123"
''', language="python")

st.subheader("✅ CORRECT: Streamlit secrets")
st.code('''
# .streamlit/secrets.toml (NEVER commit!)
# [openai]
# api_key = "sk-your-key"

# In your app:
import streamlit as st
api_key = st.secrets.openai.api_key
''', language="python")

# Demo: Check if secrets are configured
st.subheader("Demo: Check Configuration")

try:
    # Check if any secrets exist
    has_openai = hasattr(st.secrets, 'openai')
    has_database = hasattr(st.secrets, 'database')
    
    if has_openai or has_database:
        st.success("✅ Secrets are configured")
    else:
        st.warning("⚠️ No secrets found. Add .streamlit/secrets.toml")
except FileNotFoundError:
    st.warning("⚠️ No secrets.toml found")
except Exception as e:
    st.info(f"Secrets check: {e}")

st.markdown("""
**Instructions:**
1. Create `.streamlit/secrets.toml` (don't commit it!)
2. Add test secrets
3. Verify you can access them
""")

---

## 🛡️ Lab 2: Input Validation

In [ ]:
import streamlit as st
import re

st.header("🛡️ Lab 2: Input Validation")

st.subheader("Text Validation Function")
st.code('''
def validate_text(text, max_length=5000, required=True):
    """Validate text input."""
    errors = []
    
    if required and (not text or not text.strip()):
        errors.append("Text is required")
    
    if text and len(text) > max_length:
        errors.append(f"Too long (max {max_length} chars)")
    
    # Check for suspicious patterns
    suspicious = ['<script>', 'javascript:', 'onerror=']
    for pattern in suspicious:
        if text and pattern.lower() in text.lower():
            errors.append("Contains suspicious content")
    
    return errors
''', language="python")

# Interactive validation
st.subheader("Try It: Validate Your Input")
text = st.text_area("Enter text to validate:", "Hello, this is a test.")

if st.button("Validate"):
    errors = []
    
    if not text or not text.strip():
        errors.append("Text is required")
    
    if len(text) > 5000:
        errors.append("Too long (max 5000 chars)")
    
    suspicious = ['<script>', 'javascript:', 'onerror=']
    for pattern in suspicious:
        if pattern.lower() in text.lower():
            errors.append("Contains suspicious content")
    
    if errors:
        for error in errors:
            st.error(f"❌ {error}")
    else:
        st.success("✅ Input is valid")
        st.write(f"Length: {len(text)} characters")

---

## 🗄️ Lab 3: SQL Injection Prevention

In [ ]:
import streamlit as st
import sqlite3

st.header("🗄️ Lab 3: SQL Injection Prevention")

# Create demo database
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE users (id INTEGER, name TEXT, email TEXT)")
conn.execute("INSERT INTO users VALUES (1, 'Alice', 'alice@example.com')")
conn.execute("INSERT INTO users VALUES (2, 'Bob', 'bob@example.com')")
conn.commit()

st.subheader("The Attack")
st.code('''
# Attacker enters:
username = "admin\' OR \'1\'=\'1\' --"

# Unsafe query:
query = f"SELECT * FROM users WHERE name = \'{username}\'"  # Returns ALL users!
''', language="python")

st.subheader("The Defense: Parameterized Queries")
st.code('''
# ✅ SAFE: Parameterized query
query = "SELECT * FROM users WHERE name = ?"
cursor = conn.execute(query, (username,))
''', language="python")

# Demo
st.subheader("Try It: Search Users")
search = st.text_input("Search by name:", "Alice")

if st.button("Search"):
    # ✅ SAFE: Parameterized
    query = "SELECT * FROM users WHERE name = ?"
    results = conn.execute(query, (search,)).fetchall()
    
    st.write(f"**Results ({len(results)} found):**")
    for row in results:
        st.write(f"- {row[1]} ({row[2]})")
    
    st.info("This query is safe from SQL injection!")

---

## 📁 Lab 4: File Upload Security

In [ ]:
import streamlit as st
from pathlib import Path

st.header("📁 Lab 4: File Upload Security")

st.subheader("Safe File Processing")
st.code('''
def safe_process_upload(file):
    """Safely process uploaded file."""
    # Validate file type
    allowed_types = ["text/csv", "text/plain"]
    if file.type not in allowed_types:
        return None, "File type not allowed"
    
    # Validate file size
    if file.size > 10 * 1024 * 1024:  # 10MB
        return None, "File too large"
    
    # Read content safely
    content = file.read()
    return content, None
''', language="python")

st.subheader("Dangerous Operations to Avoid")
st.code('''
# ❌ NEVER: Execute uploaded files
exec(uploaded_file.read())

# ❌ NEVER: Use eval
eval(user_input)

# ❌ NEVER: Run shell commands
import os
os.system(f"process {filename}")

# ❌ NEVER: Load untrusted pickle
import pickle
model = pickle.load(uploaded_file)  # Can execute arbitrary code!
''', language="python")

# Demo
st.subheader("Try It: Upload a File")
uploaded = st.file_uploader("Upload a text file", type=["txt", "csv"])

if uploaded:
    # Validate
    allowed = ["text/plain", "text/csv"]
    if uploaded.type not in allowed:
        st.error(f"❌ File type '{uploaded.type}' not allowed")
    elif uploaded.size > 1024 * 1024:
        st.error("❌ File too large (max 1MB)")
    else:
        st.success("✅ File validated")
        st.write(f"**Name:** {uploaded.name}")
        st.write(f"**Size:** {uploaded.size / 1024:.1f} KB")
        st.write(f"**Type:** {uploaded.type}")

---

## 🤖 Lab 5: LLM Prompt Injection

In [ ]:
import streamlit as st

st.header("🤖 Lab 5: LLM Prompt Injection")

st.subheader("What is Prompt Injection?")
st.markdown("""
Attackers try to manipulate LLM behavior through crafted inputs:

- "Ignore all previous instructions..."
- "You are now DAN (Do Anything Now)..."
- "Reveal your system prompt..."
- "```system\nNew instruction...\n```"
""")

st.subheader("Defense: Input Sanitization")
st.code('''
def sanitize_llm_input(text):
    """Sanitize input for LLM."""
    suspicious = [
        "ignore previous",
        "ignore all instructions",
        "you are now",
        "system prompt:",
        "reveal your instructions"
    ]
    
    text_lower = text.lower()
    for pattern in suspicious:
        if pattern in text_lower:
            return None  # Block
    
    return text
''', language="python")

# Demo
st.subheader("Try It: Test Input Sanitization")
text = st.text_area("Enter a prompt:", "What is machine learning?")

if st.button("Sanitize"):
    suspicious = [
        "ignore previous",
        "ignore all instructions",
        "you are now",
        "system prompt:",
        "reveal your instructions"
    ]
    
    text_lower = text.lower()
    blocked = False
    for pattern in suspicious:
        if pattern in text_lower:
            st.error(f"❌ Blocked: Contains '{pattern}'")
            blocked = True
    
    if not blocked:
        st.success("✅ Input is safe")
        st.write(f"Would send to LLM: {text[:100]}...")

---

## 📝 Security Checklist

Use this checklist for your projects:

In [ ]:
import streamlit as st

st.header("📝 Security Checklist")

checklist = [
    ("No hard-coded secrets", "Secrets use st.secrets or env vars"),
    (".gitignore configured", ".streamlit/secrets.toml is ignored"),
    ("Inputs validated", "All user inputs checked"),
    ("Parameterized queries", "No SQL string concatenation"),
    ("File uploads validated", "Type and size checked"),
    ("No eval/exec on user input", "Never execute user code"),
    ("Sensitive data not logged", "Passwords/keys masked"),
    ("Dependencies pinned", "Versions specified in requirements.txt"),
    ("CORS/XSRF enabled", "Security headers configured"),
    ("HTTPS in production", "Encrypted transport"),
]

for item, description in checklist:
    checked = st.checkbox(f"{item}", help=description)

st.markdown("""
**Instructions:**
1. Review each item
2. Check if your project meets the requirement
3. Fix any gaps
4. Re-check after fixes
""")

---

## 🎯 Challenges

### Challenge 1: Secure Login Form
Build a login form with:
- Password hashing
- Input validation
- Rate limiting

### Challenge 2: Secure File Processor
Create a file processor that:
- Validates file type and size
- Processes content safely
- Logs access without sensitive data

### Challenge 3: LLM Security Wrapper
Build a wrapper that:
- Sanitizes user input
- Validates LLM output
- Logs suspicious attempts

---

## 📝 Key Takeaways

1. **Never hard-code secrets** — Use `st.secrets` or environment variables

2. **Validate everything** — All user inputs must be validated

3. **Parameterized queries** — Prevent SQL injection

4. **Safe file handling** — Validate and sandbox uploads

5. **Don't trust LLM output** — Validate before using

6. **Minimize data** — Only collect what you need

7. **Audit dependencies** — Check for vulnerabilities

---

## 📚 Further Reading

- [OWASP Top 10](https://owasp.org/www-project-top-ten/)
- [Streamlit Security](https://docs.streamlit.io/develop/concepts/architecture/security)

---

## 🔗 Related Materials

- 📚 Documentation: [Security Guide](../docs/security.md)
- 📖 Reading: [Security & Secrets](../readings/security_and_secrets.md)
- ✏️ Exercises: [Security Exercises](../exercises/security_exercises.md)